# Update a FIAT Model

This example demonstrates how to update an existing Delft-FIAT model using
the HydroMT-FIAT Python API. Updating a model allows you to add new data,
change configurations, or modify existing components without rebuilding from scratch.

In [ ]:
# Imports
import tempfile
from pathlib import Path

from hydromt import log

from hydromt_fiat import FIATModel
from hydromt_fiat.data import fetch_data

log.initialize_logging()

## Step 1: Build an initial model

First, let's build a basic model that we can then update.

In [ ]:
# Get data
global_dir = fetch_data("global-data")
local_dir = fetch_data("test-build-data")

# Create a temporary directory for our model
temp_dir = tempfile.TemporaryDirectory(delete=False)
model_dir = Path(temp_dir.name)

# Build initial model
model = FIATModel(
    root=model_dir,
    mode="w+",
    data_libs=[
        Path(global_dir, "data_catalog.yml"),
        Path(local_dir, "data_catalog.yml"),
    ],
)

# Setup the base model
model.setup_config(
    model_type="geom", calculation_method="flood.depth", **{"output.path": "output"}
)
model.setup_region(Path(local_dir, "region.geojson"))

model.vulnerability.setup(
    vulnerability_fname="jrc_curves",
    vulnerability_linking_fname="jrc_curves_link",
    unit="m",
    continent="europe",
)

model.hazard.setup(hazard_fnames="flood_event", hazard_type="water depth")

model.exposure_geoms.setup(
    exposure_fname="osm_buildings",
    exposure_object_type_column="building",
    exposure_link_fname="osm_buildings_link",
    exposure_object_type_fill="unknown",
)

# Write the initial model
model.write()
print(f"Initial model written to: {model_dir}")

## Step 2: Open the model in read+ mode

To update an existing model, open it in `r+` mode. This reads the existing
data and allows modifications.

In [ ]:
# Open in read+ mode for updating
model = FIATModel(
    root=model_dir,
    mode="r+",
    data_libs=[
        Path(global_dir, "data_catalog.yml"),
        Path(local_dir, "data_catalog.yml"),
    ],
)
model.read()
print("Model loaded successfully.")
print(f"Existing exposure features: {len(model.exposure_geoms.data)}")

## Step 3: Link vulnerability curves

Now let's update the model by linking the exposure to vulnerability curves
and adding maximum damage values.

In [ ]:
# Link exposure to vulnerability
model.exposure_geoms.setup_link_vulnerability(
    exposure_name="osm_buildings",
    impact_type="damage",
)
print("Vulnerability linked.")

# Add maximum damage values
model.exposure_geoms.setup_max_damage(
    exposure_name="osm_buildings",
    impact_type="damage",
    exposure_cost_table_fname="jrc_damage",
    country="World",
)
print("Max damage values added.")

## Step 4: Add elevation data

We can also update specific columns in the exposure data, for example
setting building elevation.

In [ ]:
# Set elevation to 0 for all buildings
model.exposure_geoms.update_column(
    exposure_name="osm_buildings",
    columns=["elevatio"],
    values=[0],
)
print("Elevation column updated.")

## Step 5: Write the updated model

In [ ]:
# Write updates
model.write()
print(f"Updated model written to: {model_dir}")

# Show the updated exposure columns
for name, gdf in model.exposure_geoms.data.items():
    print(f"\nExposure '{name}' columns: {list(gdf.columns)}")
    print(gdf.head())

## Summary

The update workflow follows these steps:

1. Open an existing model with `mode="r+"`
2. Read the model with `model.read()`
3. Call setup methods to add/modify data
4. Write the model with `model.write()`

This is equivalent to using the CLI with:
```bash
hydromt update fiat ./modeldata -i update_recipe.yml -d data_catalog.yml
```